# CairoGo — Phase 1.3: Shared Data Cleaning Pipeline

**Team:** Rawda Raafat (D) · Mariam Ehab (A) · Rahma Bahgat (B) · Bassant Tarek (C)  
**Branch:** `phase1/data-cleaning`  
**Input:** `data/raw/CairoGo_pack_v17/trips.csv` (original, untouched)  
**Output:** `data/processed/trips_model_ready.csv`

---

### Pipeline Execution Order

```
Original Dataset (df_original)
        │
        ▼  Section 2 — Person A: Fix Audit Issues
        ▼  Section 3 — Person B: Handle Missing Values
        ▼  Section 4 — Person C: Fix Data Types & Duplicates
        ▼  Section 5 — Person D: Validate Target
        │
   df_model_ready  →  trips_model_ready.csv
```

---
## Section 1 — Setup & Load Raw Data

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PACK = ROOT / "data" / "raw" / "CairoGo_pack_v17"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

df_original = pd.read_csv(PACK / "trips.csv")
zones_df    = pd.read_csv(PACK / "zones.csv")
cust_df     = pd.read_csv(PACK / "customers.csv")
drivers_df  = pd.read_csv(PACK / "drivers.csv")

df_clean = df_original.copy()

print(f"Loaded trips : {df_original.shape}")

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\CairoGo_pack_v17\\trips.csv'

---
## Section 2 — Mariam Ehab (Person A): Fix Audit Issues
**Owner:** Mariam Ehab Hassan  
Remaps legacy zone names (`Dokki`, `New Cairo`, `Badr City`) in `pickup_zone` and `dropoff_zone` to official names in `zones.csv`.

In [ ]:
from src.data_corrections import apply_all_corrections

n_before = df_clean["pickup_zone"].isin(["Dokki", "New Cairo", "Badr City"]).sum()
print(f"Rows with legacy zone names before fix : {n_before}")

df_clean = apply_all_corrections(df_clean, zones_df)

n_after = df_clean["pickup_zone"].isin(["Dokki", "New Cairo", "Badr City"]).sum()
print(f"Rows with legacy zone names after fix  : {n_after}")
print(f"Rows retained                          : {len(df_clean)} / {len(df_original)}")

---
## Section 3 — Person B: Handle Missing Values
**Owner:** Rahma Bahgat  
**Status:** ⬜ TODO

---
## Section 4 — Person C: Fix Data Types & Duplicates
**Owner:** Bassant Tarek  
**Status:** ✅ Done  
Standardizes identifier (`string`), datetime (`request_ts`), numeric (`float64`/`int64`), and categorical dtypes, removes exact duplicate rows, and verifies the resulting schema via `src/data_type_duplicate.py`.

In [ ]:
from src.data_type_duplicate import fix_dtypes, drop_duplicates, verify_types, check_duplicate_ids

print("dtypes before:")
print(df_clean.dtypes)

df_clean = fix_dtypes(df_clean)
df_clean = drop_duplicates(df_clean)

print("\ndtypes after:")
print(df_clean.dtypes)

verify_types(df_clean)
check_duplicate_ids(df_clean, id_col="trip_id", raise_on_duplicates=True)
print(f"Rows retained: {len(df_clean)} / {len(df_original)}")

---
## Section 5 — Person D: Validate Target
**Owner:** Rawda Raafat Ramadan  
**Status:** ⬜ TODO

---
## Section 6 — Final Validation & Export (All Members)

In [ ]:
print("=" * 50)
print("FINAL CLEANED DATASET SUMMARY")
print("=" * 50)
print(f"Shape             : {df_clean.shape}")
print(f"Remaining missing : {df_clean.isnull().sum().sum()}")
print(f"Remaining dups    : {df_clean.duplicated().sum()}")
print(df_clean["request_failed"].value_counts(dropna=False))

LEAKAGE_COLS = [
    "actual_trip_time_min",
    "cancel_reason",
    "driver_arrival_min",
    "support_ticket_flag",
]
ID_COLS = ["trip_id"]
drop_cols = [c for c in LEAKAGE_COLS + ID_COLS if c in df_clean.columns]
df_model = df_clean.drop(columns=drop_cols)

df_clean.to_csv(PROCESSED / "trips_cleaned.csv", index=False)
df_model.to_csv(PROCESSED / "trips_model_ready.csv", index=False)
print(f"Saved trips_cleaned.csv ({df_clean.shape}) and trips_model_ready.csv ({df_model.shape})")